GRU - is a gated recurrent unit - an analog of LSTM with same functional. Only differance is, that GRU has less parameters (gates) and uses less memory for calculation operations.

In [2]:
import torch 
from torch import nn

In [3]:
conv_block = nn.Sequential(
    nn.Conv2d(in_channels=3, out_channels=32, kernel_size=3, stride=2),
    nn.Conv2d(in_channels=32, out_channels=64, kernel_size=3, stride=2),
    nn.Conv2d(in_channels=64, out_channels=128, kernel_size=3, stride=2),
    nn.Conv2d(in_channels=128, out_channels=256, kernel_size=3, stride=2),
    nn.Conv2d(in_channels=256, out_channels=512, kernel_size=3, stride=2),
    nn.Conv2d(in_channels=512, out_channels=1024, kernel_size=3, stride=2)
)

In [4]:
# to extract visual features, before passing through GRU block, tensor should be passed through
# convolutional blocks.

# imagine like we've got a video clip with 16, of size 128 x 128 and in RGB-format, let it be a tensor:
tensor = torch.randn(1, 16, 3, 128, 128)
B, F, C, H, W = tensor.shape
# B - batch
# F - frame
# C - channel
# H, W - height, width
out = tensor.view(B*F, C, H, W)
out = conv_block(out)
out = out.flatten(1)
out = out.view(B, F, -1)
out.shape # -> torch.Size([1, 16, 1024]) - this already could be passed through LSTM

torch.Size([1, 16, 1024])

In [7]:
B, F, C = out.shape # -> [1, 16, 1024]

gru = nn.GRU(
    input_size=C,
    hidden_size=512,
    num_layers=1,
    batch_first=True,
    bidirectional=True,
)

output, h_c = gru(out)
print(output.shape, h_c.shape)

# GRU such as LSTM block in PyTorch is not only a block, but a sequence of GRU cells. So if we've got 16 frames in video, we will have 16 cells of GRU in a network

# -> (torch.Size([1, 16, 1024]), torch.Size([2, 1, 512]))

# where output - is a collection of hidden states of each frame - thats why it has 16 frames. 
# It also has a size of 1024 (512 * 2) because block is bidirectional and contains forward and back states 

# h_c - is a hidden state of final cell

# both of them have size of [num_layers * num_directions, B, hidden_size]


# after all operation, just put a mean value of hidden collection to pass it through following layers
output = output.mean(1)
print(output.shape)

torch.Size([1, 16, 1024]) torch.Size([2, 1, 512])
torch.Size([1, 1024])
